# 01 Subspace Representability

This notebook solves every saved statevector QSE row with a fixed QSE stable dimension. Errors are calculated by taking the best match with the PYSCF energies, assuming that some roots will be missing. 

In [48]:
%load_ext autoreload
%autoreload 2

import itertools
from types import SimpleNamespace

import numpy as np
import pandas as pd

from qibochem.ansatz.ucc import (Ansatz_UCCGSD, Ansatz_UCCSD, Ansatz_UCCSDSinglet,
                                    Ansatz_kUpCCGSDSinglet)

from config import DATA_DIR, PROCESSED_DATAFRAMES_DIR, PROJECT_ROOT
from notebook_utils.latex import ground_state2tex
from notebook_utils.mat_fn import solve_energy_spin_dim
from notebook_utils.general import parse_active_space, read_jsonl
from notebook_utils.operators import get_best_pairing, get_stable_states

SV_DATAFRAME_PATH = PROCESSED_DATAFRAMES_DIR / "sv_qse_data.pkl"
QSE_STATE_MATRICES_PATH = DATA_DIR / "raw_matrices" / "qse_state_matrices.jsonl"

df_sv = pd.read_pickle(SV_DATAFRAME_PATH)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
# Define QSE State matrices, used to project wavefunctions in basis of expansion coefficients
# onto second quantised determinant basis 
def load_qse_state_matrices(path):
    qse_state_matrices = {}
    
    for row in read_jsonl(path):
            key = (
                row["molecule"],
                row["active_space"],
                row["ansatz"],
                row["expansion"],
            )

            qse_state_matrices[key] = np.asarray(
                row["qse_state_matrix"],
                dtype=float,
            )
    return qse_state_matrices

qse_state_matrices = load_qse_state_matrices(QSE_STATE_MATRICES_PATH)

In [50]:
# Get the projection value for each of the exact pyscf roots onto the QSE subspace
def project_exact_roots(qse_subspace, pyscf_casci_pvec):
    qse_subspace = np.asarray(qse_subspace, dtype=complex)
    exact_root_projections = {}

    for root_label, exact_pvec in pyscf_casci_pvec.items():
        exact_pvec = np.asarray(exact_pvec, dtype=complex)
        exact_norm = np.linalg.norm(exact_pvec)

        if exact_norm == 0:
            exact_root_projections[root_label] = 0.0
            continue

        exact_pvec = exact_pvec / exact_norm
        projection_coeffs = qse_subspace.conj().T @ exact_pvec
        projection_weight = np.sum(np.abs(projection_coeffs) ** 2)

        exact_root_projections[root_label] = float(np.real_if_close(projection_weight))

    return exact_root_projections


In [51]:
# Group triplet energies and exact-state projections
# Record the maximum sector splitting for both quantities
TRIPLET_SECTORS = ("0", "p1", "m1")

def group_triplet_energies(values):
    """
    Group triplet energies that are in a list [E00, E0p1, E0m1, E10, E1p1, E1m1 ...]
    """
    values = np.asarray(values, dtype=float)
    sector_energies = values.reshape(-1, 3)

    grouped_energies = sector_energies.mean(axis=1).tolist()
    energy_splittings = np.ptp(sector_energies, axis=1).tolist()

    return grouped_energies, energy_splittings


def group_triplet_projections(projection):
    """
    Group triplet projection values from a dictionary of the form 
    {"0_0": ..., "0_p1": ..., "0_m1": ..., ...}
    """
    num_roots = len(projection) // 3
    # Re shape dict into array
    sector_projections = np.asarray([
        [projection[f"{root}_{sector}"] for sector in TRIPLET_SECTORS]
        for root in range(num_roots)
    ], dtype=float)

    grouped_projections = {
        str(root): float(mean_projection)
        for root, mean_projection in enumerate(sector_projections.mean(axis=1))
    }
    projection_splittings = np.ptp(sector_projections, axis=1).tolist()

    return grouped_projections, projection_splittings


In [52]:
# Define spin contamination here
def get_spin_contamination(spin_values, expansion):
    spin_values = np.asarray(spin_values, dtype=float)

    if expansion == "singlet":
        return np.abs(spin_values).tolist()

    if expansion == "triplet_all":
        triplet_spins = spin_values.reshape(-1, 3)
        return np.max(np.abs(triplet_spins - 2.0), axis=1).tolist()

# Match individaul spin to the best paring roots 
def match_spin_contamination(best_energy_pairs, qse_energies, spin_contamination):
    qse_spin = dict(zip(qse_energies, spin_contamination))

    return {
        str(root): np.nan if qse_energy is None else qse_spin[qse_energy]
        for root, (_, qse_energy) in enumerate(best_energy_pairs)
    }


In [53]:
# Solve every QSE row and add on the energies, spin contamination and subspace
def solve_subspace_row(row):
    num_e, num_o = parse_active_space(row["active_space"])
    singlet_dim, triplet_dim = get_stable_states((num_e, num_o))
    dim = {"singlet": singlet_dim, "triplet_all": triplet_dim}[row["expansion"]]
    dim = min(dim, row["S"].shape[0])

    qse_energies, qse_wfn, spin_values = solve_energy_spin_dim(row["H"],
                                                               row["S"],
                                                               row["S2"],
                                                               dim)
    spin_contamination = get_spin_contamination(spin_values, row["expansion"])
    
    # Get the QSE state matrix for subspace calculation
    key = (row["molecule"], row["active_space"], row["ansatz"], row["expansion"])
    qse_state_matrix = qse_state_matrices[key]
    qse_subspace = qse_state_matrix @ qse_wfn
    
    # Calculate the projection of the exact roots 
    exact_qse_projection = project_exact_roots(qse_subspace, 
                                               row["pyscf_casci_pvec"])
    energy_splittings = []
    projection_splittings = []

    # Use the above functions to group the triplet terms
    if row["expansion"] == "triplet_all":
        qse_energies, energy_splittings = group_triplet_energies(qse_energies)
        exact_qse_projection, projection_splittings = group_triplet_projections(
            exact_qse_projection
        )

    best_energy_pairs = get_best_pairing(row["pyscf_casci_energies"], qse_energies)
    spin_contamination = match_spin_contamination(best_energy_pairs, qse_energies, spin_contamination)

    return pd.Series({
        "qse_energies": qse_energies,
        "spin_contamination": spin_contamination,
        "qse_subspace": qse_subspace,
        "exact_qse_projection": exact_qse_projection,
        "best_energy_pairs": best_energy_pairs,
        "energy_splittings": energy_splittings,
        "projection_splittings": projection_splittings,
    })


result_columns = [
    "qse_energies",
    "spin_contamination",
    "qse_subspace",
    "exact_qse_projection",
    "best_energy_pairs",
    "energy_splittings",
    "projection_splittings",
]

df_sv[result_columns] = df_sv.apply(solve_subspace_row, axis=1)

max_triplet_energy_splitting = float(df_sv["energy_splittings"].explode().max())
max_triplet_projection_splitting = float(df_sv["projection_splittings"].explode().max())

df_sv.drop(columns=["energy_splittings", "projection_splittings"], inplace=True)

print(f"Max triplet QSE energy splitting: {max_triplet_energy_splitting:.12e} Ha")
print(f"Max triplet projection splitting: {max_triplet_projection_splitting:.12e}")

df_sv.head()

Max triplet QSE energy splitting: 9.478891516608e-04 Ha
Max triplet projection splitting: 2.587910136708e-02


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec,qse_energies,spin_contamination,qse_subspace,exact_qse_projection,best_energy_pairs
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5663414572622+0j), (-4.706692634862637...","[[(3.9097041888243123+0j), (0.0229245233142809...",4,"[[(1.27675647831893e-14+0j), (-4.3368086899420...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098...","[-205.27548280391832, -204.8233864545022]","{'0': 3.2652541364035303e-15, '1': 1.276322327...","[[(4.715955244715434e-05+0j), (-0.007790346367...","{'0': 0.9774731013374688, '1': 0.9938579663436...","[(-205.29449911826808, -205.27548280391832), (..."
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet_all,"[[(-0.008069504761204022+0j), 0j, 0j, (0.19305...","[[(3.934912814912428e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.869825629530647e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067...",[-205.07455033364008],{'0': 4.884981308350689e-15},"[[(-7.272339806157963e-33+0j), (-1.13283213477...",{'0': 0.9999999994616641},"[(-205.07455054622508, -205.07455033364008)]"
2,Acetamide,2e2o,UCCGSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098...","[-205.27548283692684, -204.82338563600212]","{'0': 3.3149329374208578e-15, '1': 1.290206356...","[[(4.7319825018137736e-05+0j), (-0.00780357810...","{'0': 0.9774731013010415, '1': 0.9938558985331...","[(-205.29449911826808, -205.27548283692684), (..."
3,Acetamide,2e2o,UCCGSD,triplet_all,"[[(-0.008087395184562356+0j), 0j, 0j, (0.19324...","[[(3.943636678180318e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.887273356062263e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067...",[-205.07455033364008],{'0': 4.884981308350689e-15},"[[(-2.528793532251224e-16+0j), (4.639168275980...",{'0': 0.9999999999349489},"[(-205.07455054622508, -205.07455033364008)]"
4,Acetamide,2e2o,UCCSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098...","[-205.27548283692684, -204.82338563600212]","{'0': 3.3149329374208578e-15, '1': 1.290206356...","[[(4.7319825018137736e-05+0j), (-0.00780357810...","{'0': 0.9774731013010415, '1': 0.9938558985331...","[(-205.29449911826808, -205.27548283692684), (..."


In [54]:
# Separate all the metrics into different individual columns
state_labels = [f"s{i}" for i in range(10)] + [f"t{i}" for i in range(1, 11)]
metrics = ("error", "projection", "spin_contamination")
metric_columns = [f"{state}_{metric}" 
                  for state in state_labels 
                  for metric in metrics]


def add_root_metrics(row):
    prefix = "s" if row["expansion"] == "singlet" else "t"
    start = 0 if row["expansion"] == "singlet" else 1

    # For singlets we start with root 0 (ground)
    # For triplets start with root 1 instead 
    for root_index, (exact_energy, qse_energy) in enumerate(row["best_energy_pairs"]):
        # Enumerate all the pairs found
        root = f"{prefix}{root_index + start}"
        # Report the error in mHa 
        row[f"{root}_error"] = np.nan if qse_energy is None else 1000 * abs(qse_energy - exact_energy)
        row[f"{root}_projection"] = row["exact_qse_projection"][str(root_index)]
        row[f"{root}_spin_contamination"] = row["spin_contamination"][str(root_index)]

    return row


df_sv[metric_columns] = np.nan
df_sv = df_sv.apply(add_root_metrics, axis=1)

df_sv.head()


,molecule,active_space,ansatz,expansion,H,S,qse_dim,S2,pyscf_casci_energies,pyscf_casci_pvec,...,t7_spin_contamination,t8_error,t8_projection,t8_spin_contamination,t9_error,t9_projection,t9_spin_contamination,t10_error,t10_projection,t10_spin_contamination
0,Acetamide,2e2o,1UpCCGSDSinglet,singlet,"[[(-802.5663414572622+0j), (-4.706692634862637...","[[(3.9097041888243123+0j), (0.0229245233142809...",4,"[[(1.27675647831893e-14+0j), (-4.3368086899420...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Acetamide,2e2o,1UpCCGSDSinglet,triplet_all,"[[(-0.008069504761204022+0j), 0j, 0j, (0.19305...","[[(3.934912814912428e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.869825629530647e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Acetamide,2e2o,UCCGSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Acetamide,2e2o,UCCGSD,triplet_all,"[[(-0.008087395184562356+0j), 0j, 0j, (0.19324...","[[(3.943636678180318e-05+0j), 0j, 0j, (-0.0009...",12,"[[(7.887273356062263e-05+0j), 0j, 0j, (-0.0018...",[-205.07455054622508],"{'0_0': [-4.001328838134542e-16, 0, -0.7071067...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Acetamide,2e2o,UCCSD,singlet,"[[(-802.5658955566977+0j), (-4.714686911530124...","[[(3.909702017207626+0j), (0.02296346038353209...",4,"[[(1.2961853812498703e-14+0j), (-4.33680868994...","[-205.29449911826808, -204.8258180441668, -204...","{'0': [-0.15005494805049122, 0, -0.00628487098...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Ansatz Details

In [55]:
# Compare the number of parameters and circuit depth for each ansatz
ANSATZ_FUNCTIONS = {
    "1UpCCGSDSinglet": lambda molecule, **kwargs: Ansatz_kUpCCGSDSinglet(molecule, k=1, **kwargs),
    "UCCSDSinglet": Ansatz_UCCSDSinglet,
    "UCCSD": Ansatz_UCCSD,
    "UCCGSD": Ansatz_UCCGSD,
}


def get_ansatz_complexity(active_space, ansatz_name):
    num_e, num_o = parse_active_space(active_space)
    molecule = SimpleNamespace(nelec=num_e, nso=2 * num_o, n_active_e=None, n_active_orbs=None)
    ansatz = ANSATZ_FUNCTIONS[ansatz_name](molecule, use_mp2_guess=False)

    return len(ansatz.param_names), ansatz.circuit.depth


ansatz_complexity = []

for active_space in sorted(df_sv["active_space"].unique(), key=parse_active_space):
    for ansatz_name in ANSATZ_FUNCTIONS:
        num_parameters, circuit_depth = get_ansatz_complexity(active_space, ansatz_name)
        ansatz_complexity.append({
            "active_space": active_space,
            "ansatz": ansatz_name,
            "num_parameters": num_parameters,
            "circuit_depth": circuit_depth,
        })

df_ansatz_complexity = pd.DataFrame(ansatz_complexity)
df_ansatz_complexity


,active_space,ansatz,num_parameters,circuit_depth
0,2e2o,1UpCCGSDSinglet,2,115
1,2e2o,UCCSDSinglet,2,115
2,2e2o,UCCSD,3,115
3,2e2o,UCCGSD,3,115
4,2e3o,1UpCCGSDSinglet,6,352
5,2e3o,UCCSDSinglet,5,433
6,2e3o,UCCSD,8,430
7,2e3o,UCCGSD,15,936
8,4e3o,1UpCCGSDSinglet,6,352
9,4e3o,UCCSDSinglet,5,440


## Ground State Comparison

In [56]:
def mean_and_max_difference(values):
    values = np.asarray(values, dtype=float)

    if len(values) == 0:
        return np.nan, np.nan

    mean_value = float(np.mean(values))
    max_difference = float(np.max(np.abs(values - mean_value)))
    return mean_value, max_difference


complexity_lookup = df_ansatz_complexity.set_index(["active_space", "ansatz"])
ground_state_tables = {}

for active_space in sorted(df_sv["active_space"].unique(), key=parse_active_space):
    df_sv_space = df_sv[(df_sv["active_space"] == active_space) &
                        (df_sv["expansion"] == "singlet")]
    ground_state_rows = []

    for ansatz_name in ANSATZ_FUNCTIONS:
        df_ansatz = df_sv_space[df_sv_space["ansatz"] == ansatz_name]
        recovered = df_ansatz[df_ansatz["s0_error"].notna()]
        complexity = complexity_lookup.loc[(active_space, ansatz_name)]

        ground_state_rows.append({
            "ansatz": ansatz_name,
            "num_parameters": complexity["num_parameters"],
            "circuit_depth": complexity["circuit_depth"],
            "mean_error": mean_and_max_difference(recovered["s0_error"]),
            "mean_projection": mean_and_max_difference(recovered["s0_projection"]),
            "spin_contamination": mean_and_max_difference(recovered["s0_spin_contamination"]),
        })

    ground_state_tables[active_space] = pd.DataFrame(ground_state_rows).set_index("ansatz")

GROUND_STATE_TABLE_DIR = PROJECT_ROOT / "manuscript" / "tables" / "ground_state_comparison"
for active_space, table in ground_state_tables.items():
    ground_state2tex(
        table,
        f"Ground-state ansatz comparison for the {active_space} active space.",
        GROUND_STATE_TABLE_DIR / f"{active_space}.tex",
        label=f"tab:ground-state-{active_space}",
    )


## Spin Analysis